# ❤️ Heart Disease Prediction - Model Training
## Explainable AI (XAI) Based Multi-Disease Risk Prediction System
### Dataset: UCI Heart Disease Dataset (920 samples, 15 features, Multi-center)

**Target:** `num` → 0 = No Disease, 1-4 = Disease (binarized: 0 vs 1+)  
**Models:** Logistic Regression, Decision Tree, Random Forest, XGBoost  
**XAI:** SHAP Values  
**Output:** `heart_model.pkl`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score
)
import shap

os.makedirs('../backend/models', exist_ok=True)
print('✅ Libraries loaded')

## 1️⃣ Load & Explore Data

In [ ]:
df = pd.read_csv('heart_disease_uci.csv')  # Place in notebooks/ folder
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(10)

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print('\nTarget Distribution:')
print(df['num'].value_counts())

In [ ]:
# Target distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['num'].value_counts().plot(kind='bar', ax=axes[0], color=['#00b4d8','#e63946','#f4a261','#2ec4b6','#6c757d'])
axes[0].set_title('Original Target (0-4)', fontweight='bold')
axes[0].tick_params(rotation=0)

binary_target = (df['num'] > 0).astype(int)
binary_target.value_counts().plot(kind='bar', ax=axes[1], color=['#00b4d8','#e63946'])
axes[1].set_title('Binarized Target (0=No Disease, 1=Disease)', fontweight='bold')
axes[1].tick_params(rotation=0)
plt.tight_layout()
plt.savefig('../backend/models/heart_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 2️⃣ Preprocessing

In [ ]:
# Drop id and dataset columns
df = df.drop(columns=['id','dataset'], errors='ignore')

# Binarize target
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns=['num'])

# Encode categorical features
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {cat_cols}')

le = LabelEncoder()
for col in cat_cols:
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])

# Impute missing values
FEATURE_NAMES = [c for c in df.columns if c != 'target']
X = df[FEATURE_NAMES]
y = df['target']

imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=FEATURE_NAMES)

print(f'Features: {FEATURE_NAMES}')
print(f'\nFinal shape after imputation: {X_imp.shape}')
print(f'\nClass balance: {y.value_counts().to_dict()}')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr_df = X_imp.copy()
corr_df['target'] = y.values
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, square=True, annot_kws={'size':7})
plt.title('Heart Disease Feature Correlation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/heart_corr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES)
X_test_sc  = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3️⃣ Model Training

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(C=0.5, max_iter=2000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=300, max_depth=10,
                                                   min_samples_leaf=5, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                          subsample=0.8, colsample_bytree=0.8,
                                          use_label_encoder=False, eval_metric='logloss', random_state=42),
}

results = []
trained_models = {}

for name, model in models.items():
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc  if name == 'Logistic Regression' else X_test
    X_cv = X_train_sc if name == 'Logistic Regression' else X_train

    cv_scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='accuracy')
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1]

    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    f1   = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)

    trained_models[name] = {'model': model, 'X_test': X_te, 'X_train': X_tr}
    results.append({'Model': name, 'CV Accuracy': cv_scores.mean(),
                    'Test Accuracy': acc, 'AUC-ROC': auc,
                    'F1 Score': f1, 'Precision': prec, 'Recall': rec})
    print(f'✅ {name}: Acc={acc:.4f} | AUC={auc:.4f} | F1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)
results_df

In [ ]:
# ROC Curves
plt.figure(figsize=(8,6))
for name, data in trained_models.items():
    y_prob = data['model'].predict_proba(data['X_test'])[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curves - Heart Disease', fontsize=14, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../backend/models/heart_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, data) in zip(axes, trained_models.items()):
    y_pred = data['model'].predict(data['X_test'])
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Reds',
                xticklabels=['No HD','HD'], yticklabels=['No HD','HD'])
    ax.set_title(name, fontweight='bold')
plt.suptitle('Confusion Matrices - Heart Disease', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/heart_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 4️⃣ SHAP Explainability

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]['model']
X_test_best = trained_models[best_name]['X_test']
X_train_best = trained_models[best_name]['X_train']

print(f'🏆 Best Model: {best_name}')

if best_name in ['XGBoost','Random Forest','Decision Tree']:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test_best)
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values
else:
    explainer = shap.LinearExplainer(best_model, X_train_best)
    shap_values = explainer.shap_values(X_test_best)
    shap_vals = shap_values

plt.figure(figsize=(10,6))
shap.summary_plot(shap_vals, X_test_best, feature_names=FEATURE_NAMES, show=False)
plt.title(f'SHAP Summary - {best_name} (Heart Disease)', fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/heart_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8,5))
shap.summary_plot(shap_vals, X_test_best, feature_names=FEATURE_NAMES, plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance - {best_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('../backend/models/heart_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5️⃣ Save Model

In [ ]:
artifact = {
    'model': best_model,
    'model_name': best_name,
    'scaler': scaler,
    'imputer': imputer,
    'feature_names': FEATURE_NAMES,
    'model_results': results_df.to_dict('records'),
    'disease': 'heart',
    'target_col': 'target',
    'classes': ['No Heart Disease', 'Heart Disease'],
    'threshold': 0.5,
    'cat_cols': cat_cols
}

with open('../backend/models/heart_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print(f'✅ Saved → backend/models/heart_model.pkl')
print(f'🏆 Best: {best_name}')
print(results_df.to_string(index=False))